# SocialShield: can a transformer read scams better?

Our message model (TF-IDF + Logistic Regression) only matches words. A **transformer** reads whole sentences, so it
should catch scams written in new words. This notebook fine-tunes **multilingual DistilBERT** (it also knows Hindi)
and compares it with our two existing detectors on the **same held-out test sets**.

**Before you run:** Runtime > Change runtime type > **T4 GPU** > Save. Then **Runtime > Run all**.
It takes about 10 to 15 minutes. At the end a file `transformer_results.json` downloads: send it to Claude.

| Detector | What it is |
|---|---|
| v2 | 37 rules + link checker + TF-IDF model trained on synthetic posts (the deployed app) |
| v3 | v2's layers + a TF-IDF model trained on synthetic posts **and real SMS**, weights learned from data |
| Transformer | multilingual DistilBERT fine-tuned on the same training data as v3 |

In [ ]:
import os, sys, json, time
import torch
assert torch.cuda.is_available(), "No GPU: go to Runtime > Change runtime type > T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))
if not os.path.exists("socialshield"):
    !git clone -q https://github.com/spy-ka-y/socialshield.git
os.chdir("socialshield")
sys.path.insert(0, "src")
!pip -q install -r requirements.txt
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd

## 1. Build the two existing detectors in this notebook
So all three are measured in exactly the same place. Takes about 2 minutes.

In [ ]:
!python src/train_post_model.py > /dev/null
!python src/train_v3.py | tail -6

## 2. Training and test data
Exactly the same split as v3. **Training:** 4,000 synthetic posts + 4,459 real SMS + the 62 Test 1 DMs.
**Testing (never trained on):** 1,115 real SMS, the 30 unseen Test 2 DMs, and posts from unseen Kaggle accounts.

In [ ]:
from train_v3 import load_sets, SEED
k_tr, k_te, s_tr, s_te, t1, t2 = load_sets()
train = pd.concat([k_tr.sample(4000, random_state=SEED)[["content", "y"]], s_tr[["content", "y"]]]
                  + [t1[["content", "y"]]] * 10,                    # the few Indian DMs count 10 times
                  ignore_index=True).sample(frac=1, random_state=SEED)
tests = {"Real SMS test split": s_te, "Test 2 (unseen DMs)": t2,
         "Kaggle test accounts": k_te.drop_duplicates("content").sample(3000, random_state=SEED)}
print("training messages:", len(train), "| scam share:", round(train["y"].mean(), 3))

## 3. Fine-tune the transformer
`distilbert-base-multilingual-cased` already understands language from pre-training on Wikipedia in 104 languages.
Fine-tuning teaches it one new task: scam or not. 3 passes over the data.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL = "distilbert-base-multilingual-cased"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2).cuda()

def batches(texts, labels=None, bs=32, shuffle=False):
    idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
    for i in range(0, len(idx), bs):
        j = idx[i:i + bs]
        enc = tok([str(texts[k]) for k in j], truncation=True, max_length=128, padding=True, return_tensors="pt")
        enc = {k: v.cuda() for k, v in enc.items()}
        yield enc, (torch.tensor([labels[k] for k in j]).cuda() if labels is not None else None)

EPOCHS, LR = 3, 2e-5
texts, labels = train["content"].tolist(), train["y"].tolist()
steps = EPOCHS * ((len(texts) + 31) // 32)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler("cuda")
np.random.seed(SEED); torch.manual_seed(SEED)
for ep in range(EPOCHS):
    model.train(); total, t0 = 0.0, time.time()
    for enc, y in batches(texts, labels, shuffle=True):
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(**enc, labels=y).loss
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        total += loss.item()
    print(f"pass {ep + 1}/{EPOCHS}: average loss {total / ((len(texts) + 31) // 32):.4f} ({time.time() - t0:.0f}s)")

## 4. Test all three detectors on the held-out sets

In [ ]:
from detector import MessageDetector, SAFE
from train_v3 import layer_features
import joblib

@torch.no_grad()
def transformer_scores(texts):
    model.eval(); out = []
    for enc, _ in batches(texts, bs=64):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.softmax(model(**enc).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)

det = MessageDetector()
v3 = joblib.load("models/v3.joblib")
def rates(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {"caught": round(float(p[y == 1].mean()), 4), "false_alarms": round(float(p[y == 0].mean()), 4), "n": int(len(y))}

results, rows = {"model": MODEL, "epochs": EPOCHS, "train_size": len(train)}, []
for name, df in tests.items():
    tx, y = df["content"].tolist(), df["y"].values
    ch = "post" if name.startswith("Kaggle") else "dm"
    X, vs = layer_features(det, tx, v3["text_model"].predict_proba(tx)[:, 1], channel=ch)
    pr = transformer_scores(tx)
    preds = {"v2": [v.label != SAFE for v in vs],
             "v3": v3["combiner"].predict_proba(X)[:, 1] >= v3["threshold"],
             "Transformer": pr >= 0.5,
             "Transformer + v2 rules": (pr >= 0.5) | np.array([v.label != SAFE for v in vs])}
    results[name] = {k: rates(y, p) for k, p in preds.items()}
    if name.startswith("Test 2"):
        results["test2_details"] = [{"text": t[:90], "label": int(l), "transformer_score": round(float(s), 3)}
                                    for t, l, s in zip(tx, y, pr)]
    for k, r in results[name].items():
        rows.append({"Test set": name, "Detector": k, "Scams caught": f"{r['caught']:.0%}", "False alarms": f"{r['false_alarms']:.1%}"})
pd.DataFrame(rows).set_index(["Test set", "Detector"])

## 5. Save the results and send them to Claude

In [ ]:
with open("transformer_results.json", "w") as f:
    json.dump(results, f, indent=2)
from google.colab import files
files.download("transformer_results.json")
print("Downloaded transformer_results.json. Attach it in the chat with Claude.")